# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

# Method Choice and Why

I chose a Random Forest classifier.

My lane focuses on identifying content pages that may deserve review. This is a classification problem because the model predicts whether a page belongs to the decline-risk group defined by the proxy label.

A Random Forest was selected because:

- It can capture non-linear relationships.
- It works well with mixed content-performance signals.
- It generally requires less feature scaling than linear models.
- The starter project showed strong Random Forest performance.

The goal is not maximum complexity. The goal is to determine whether a learned model can improve prioritization compared with the Week 4 baseline rule.

In [1]:
# Load Data
import pandas as pd
import numpy as np

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

print(df.shape)

df.head()

(30000, 44)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [2]:
# Create Target
df["is_declining_label"] = (
    df["trend_direction"] == "down"
).astype(int)

df["is_declining_label"].value_counts()

is_declining_label
1    16262
0    13738
Name: count, dtype: int64

In [3]:
# Building Feature Set

X = df[
    [
        "impressions_90d",
        "clicks_90d",
        "sessions_90d",
        "ctr",
        "avg_position",
        "content_age_days"
    ]
].fillna(0)

y = df["is_declining_label"]

print(X.shape)

(30000, 6)


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

# Split Design

I used a simple train/test split.

Training data: 80%

Testing data: 20%

The baseline and model are evaluated on the same holdout set to ensure a fair comparison.

This provides an honest estimate of performance while keeping the workflow simple.

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [4]:
# Creating Train/Test Split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [5]:
#Recreating Baseline
baseline_score = (
    0.4 *
    (
        X_test["impressions_90d"]
        /
        X["impressions_90d"].max()
    )
    +
    0.3 *
    (
        X_test["content_age_days"]
        /
        X["content_age_days"].max()
    )
    +
    0.3 *
    (
        1 -
        (
            X_test["ctr"]
            /
            max(X["ctr"].max(),0.01)
        )
    )
)

In [6]:
# Measuring Baseline

from sklearn.metrics import roc_auc_score

baseline_auc = roc_auc_score(
    y_test,
    baseline_score
)

print("Baseline ROC AUC:", round(baseline_auc,4))

Baseline ROC AUC: 0.4171


In [7]:
# Training Random Forest

from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

rf.fit(
    X_train,
    y_train
)

rf_probs = rf.predict_proba(
    X_test
)[:,1]

In [8]:
#Evaluate Model
model_auc = roc_auc_score(
    y_test,
    rf_probs
)

print("Model ROC AUC:", round(model_auc,4))

Model ROC AUC: 0.7282


In [9]:
# Buildign Comparison Table
results = pd.DataFrame({
    "Method": [
        "Week 4 Baseline",
        "Random Forest"
    ],
    "ROC_AUC": [
        baseline_auc,
        model_auc
    ]
})

results

,Method,ROC_AUC
0,Week 4 Baseline,0.417110
1,Random Forest,0.728249


# Model vs Baseline

The Random Forest model was evaluated on the same test set as the baseline rule.

This comparison is important because performance numbers are only meaningful when both methods use the same data and evaluation process.

The model's objective is to improve prioritization beyond the manually designed baseline.

In [11]:
# Feature Importance
importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": rf.feature_importances_
})

importance.sort_values(
    "Importance",
    ascending=False
)

,Feature,Importance
4,avg_position,0.253947
0,impressions_90d,0.253118
5,content_age_days,0.207715
2,sessions_90d,0.128664
3,ctr,0.089528
1,clicks_90d,0.067028


In [12]:
# Error Review
from sklearn.metrics import confusion_matrix

preds = (
    rf_probs > 0.5
).astype(int)

confusion_matrix(
    y_test,
    preds
)

array([[1621, 1127],
       [ 849, 2403]])

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

# Errors and Interpretation

The model still produces false positives and false negatives.

False Positives:
Pages predicted as decline-risk that may not actually require review.

False Negatives:
Pages that deserve review but were missed by the model.

The model should be viewed as decision-support rather than an automated decision maker.

Feature importance suggests that impressions, CTR, position, and content age contribute substantially to the prediction.

## Self-check

Before you submit, confirm each line honestly:

- [ y] Every section above is filled — markdown thinking AND the code that backs it
- [ y] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ y] No client names, URLs, or private queries anywhere
- [ y] My claims use careful words: observed, measured, directional, decision-support
- [ y] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.